In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_evasao = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/02_filtered/Maiores_Taxas_Evasao_e_Reprovacao_2024.csv")
df_censo = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/01_Cleaned/Tabela_Censo_Escolar_2024.csv", sep=";")
df_enem = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/01_Cleaned/Tabela_ENEM_2024.csv")


df_evasao['evasao_medio_total'] = pd.to_numeric(df_evasao['evasao_medio_total'].replace('Não informado', np.nan), errors='coerce')
df_escolas_censo = pd.merge(df_censo, df_evasao, left_on='CO_ENTIDADE', right_on='codigo_escola', how='inner')
df_escolas_censo['taxa_permanencia'] = 100 - df_escolas_censo['evasao_medio_total']

permanencia_uf = df_escolas_censo.groupby('SG_UF')['taxa_permanencia'].mean().reset_index()
enem_uf = df_enem.groupby('SG_UF_PROVA')[['NOTA_GERAL', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']].mean().reset_index()

df_final = pd.merge(permanencia_uf, enem_uf, left_on='SG_UF', right_on='SG_UF_PROVA', how='inner')
df_final = df_final.sort_values(by='taxa_permanencia', ascending=False)


# ==========================================
# --- RANKING DE PERMANÊNCIA NAS ESCOLAS DO CENSO E NOTAS DO ENEM ---
# ==========================================
print("--- RANKING DE PERMANÊNCIA NAS ESCOLAS DO CENSO E NOTAS DO ENEM ---")

display(df_final[['SG_UF', 'taxa_permanencia', 'NOTA_GERAL', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']].head(10).round(2))
print("\n")

# ==========================================
# --- GERAÇÃO DOS GRÁFICOS () ---
# ==========================================
plt.figure(figsize=(10, 6))

sns.regplot(data=df_final, x='taxa_permanencia', y='NOTA_GERAL', color='b', scatter_kws={'s': 50})

for i, row in df_final.iterrows():
    plt.text(row['taxa_permanencia'], row['NOTA_GERAL'] + 1.5, row['SG_UF'], fontsize=9, ha='center')

media_nacional = df_enem['NOTA_GERAL'].mean()
plt.axhline(media_nacional, color='red', linestyle='--', label=f'Média Nacional ENEM ({media_nacional:.2f})')

plt.title('Taxa de Permanência Escolar vs Nota Média Geral no ENEM\n(Focado nas Escolas do Censo Filtrado)')
plt.xlabel('Taxa de Permanência do Aluno (%)')
plt.ylabel('Nota Média Geral no ENEM')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

plt.show()